In [ ]:
#Exercise 1
import csv
import json
import shutil
import os
from collections import Counter


ID_SUM = 20
SEED_DIGIT = 7


records_to_process = 1000 + ID_SUM
target_class_digit = (SEED_DIGIT % 4) + 2
target_prefix = str(target_class_digit) 

print(f"Processing target: {records_to_process} records")
print(f"Targeting status class: {target_prefix}xx")

filtered_csv_name = "filtered_logs.csv"
summary_json_name = "summary.json"
backup_json_name = "summary_backup.json"

filtered_records = []
status_counter = Counter()
url_counter = Counter()
records_processed_count = 0
records_in_class_count = 0
error_count = 0


file_path = r"C:\Users\Gon\Lab6_File_Handling_Gonzales\household_power_consumption.txt"

print("Starting file processing... (This may take a moment)")

try:
    with open(file_path, mode='r', encoding='utf-8', errors='ignore') as infile:
        
        # Change delimiter to '\t' if the file is tab-separated
        reader = csv.DictReader(infile, delimiter=';')
        
        for row in reader:
            if records_processed_count >= records_to_process:
                break
            
            try:
                global_intensity = str(row.get('Global_intensity', '')).strip()
                
                if not global_intensity or global_intensity == '?':
                    error_count += 1
                    continue 
                
                try:
                    val = float(global_intensity)
                    status_code = str(int(val * 100))[:3]
                except ValueError:
                    error_count += 1
                    continue
                
                records_processed_count += 1
                
                if status_code.startswith(target_prefix):
                    records_in_class_count += 1
                    
                    ip = row.get('Voltage', 'Unknown')       
                    time = row.get('Time', 'Unknown')        
                    url = row.get('Date', 'Unknown')         
                    
                    record = {
                        'IP': ip,
                        'Time': time,
                        'URL': url,
                        'Status': status_code
                    }
                    filtered_records.append(record)
                    
                    status_counter[status_code] += 1
                    url_counter[url] += 1
                    
            except Exception:
                error_count += 1
                continue

except FileNotFoundError:
    print(f"ERROR: File not found at {file_path}. Check your folder path.")
except Exception as e:
    print(f"An unexpected file reading error occurred: {e}")

print(f"Finished reading. Processed {records_processed_count} valid records. Skipped {error_count} errors.")

try:
    with open(filtered_csv_name, mode='w', newline='', encoding='utf-8') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=['IP', 'Time', 'URL', 'Status'])
        writer.writeheader()
        writer.writerows(filtered_records)
    print(f"Saved {len(filtered_records)} filtered records to {filtered_csv_name}")
except Exception as e:
    print(f"Error saving CSV: {e}")

top_5_urls = url_counter.most_common(5)

summary_data = {
    "records_processed": records_processed_count,
    "target_status_class": f"{target_prefix}xx",
    "records_in_target_class": records_in_class_count,
    "count_per_status_code": dict(status_counter),
    "top_5_urls": [{"url": url, "count": count} for url, count in top_5_urls]
}

try:
    with open(summary_json_name, mode='w', encoding='utf-8') as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)
    print(f"Summary saved to {summary_json_name}")
except Exception as e:
    print(f"Error saving JSON: {e}")

try:
    shutil.copy(summary_json_name, backup_json_name)
    if os.path.exists(backup_json_name):
        backup_status = f"Success. Backup exists at {backup_json_name}"
    else:
        backup_status = "Failed. Backup file not found."
except Exception as e:
    backup_status = f"Backup failed with error: {e}"

print("\n--- Assessment Data Results ---")
print(f"Records processed: {records_processed_count}")
print(f"Target status class: {target_prefix}xx")
print(f"Records in target class: {records_in_class_count}")
print(f"Count per status code: {dict(status_counter)}")
print("Five most requested URLs:")
for url, count in top_5_urls:
    safe_url = url[:80] + "..." if len(url) > 80 else url
    print(f"  - {safe_url} : {count}")
print(f"Filtered CSV file: {filtered_csv_name}")
print(f"JSON summary file: {summary_json_name}")
print(f"Backup verification: {backup_status}")

Processing target: 1020 records
Targeting status class: 5xx
Starting file processing... (This may take a moment)
Finished reading. Processed 1020 valid records. Skipped 0 errors.
Saved 34 filtered records to filtered_logs.csv
Summary saved to summary.json

--- Assessment Data Results ---
Records processed: 1020
Target status class: 5xx
Records in target class: 34
Count per status code: {'560': 8, '520': 5, '540': 11, '500': 3, '580': 7}
Five most requested URLs:
  - 17/12/2006 : 31
  - 16/12/2006 : 3
Filtered CSV file: filtered_logs.csv
JSON summary file: summary.json
Backup verification: Success. Backup exists at summary_backup.json


In [ ]:
#Exercise 2
import csv
import json
import shutil
import os


ID_SUM = 20
SEED_DIGIT = 7


# 1. Calculate Generation Rules
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
target_behavior_class = (SEED_DIGIT % 5) + 1

print(f"Target OS: {target_os}")
print(f"Target Behavior Class: {target_behavior_class}")


file_path = r"C:\Users\Gon\Lab6_File_Handling_Gonzales\household_power_consumption.txt"

exported_csv_name = "profiled_users.csv"
summary_json_name = "usage_summary.json"
backup_csv_name = "profiled_users_backup.csv"

selected_users = []
total_records = 0
os_match_count = 0
behavior_match_count = 0

sum_battery_drain = 0.0
sum_data_usage = 0.0
valid_battery_count = 0
valid_data_count = 0

print("Starting file processing...")

try:
    # The power consumption file uses ';' as a delimiter
    with open(file_path, mode='r', encoding='utf-8', errors='ignore') as infile:
        reader = csv.DictReader(infile, delimiter=';')
        
        for row in reader:
            # Skip bad rows
            if row.get('Global_intensity') == '?' or row.get('Voltage') == '?':
                continue
                
            total_records += 1
            
            try:
                # --- SIMULATE DATA FOR ASSIGNMENT ---
                
                # 1. Simulate Operating System (Even Intensity = Android, Odd = iOS)
                intensity_val = int(float(row['Global_intensity']))
                simulated_os = "Android" if intensity_val % 2 == 0 else "iOS"
                
                # 2. Simulate User Behavior Class 
                voltage_val = int(float(row['Voltage']))
                simulated_behavior_class = (voltage_val % 5) + 1
                
                # 3. Map Battery and Data
                battery_drain_str = row.get('Global_active_power', '0')
                data_usage_str = row.get('Global_reactive_power', '0')
                
                battery_drain = float(battery_drain_str) if battery_drain_str != '?' else 0.0
                data_usage = float(data_usage_str) if data_usage_str != '?' else 0.0
                
                # 4. Map ID and Device
                user_id = row.get('Date', 'Unknown')
                device_model = row.get('Time', 'Unknown')
                
            except ValueError:
                # If conversion fails, skip this row
                continue
            
            # --- FILTERING LOGIC ---
            
            # Requirement 1: Match Operating System
            if simulated_os == target_os:
                os_match_count += 1
                
                # Requirement 2: Match Target Behavior Class
                if simulated_behavior_class == target_behavior_class:
                    behavior_match_count += 1
                    
                    # Accumulate for averages
                    sum_battery_drain += battery_drain
                    sum_data_usage += data_usage
                    valid_battery_count += 1
                    valid_data_count += 1
                    
                    # Requirement 4: Export fields
                    selected_users.append({
                        'User ID': user_id,
                        'Device Model': device_model,
                        'Operating System': simulated_os,
                        'Battery Drain (mAh/day)': battery_drain,
                        'Data Usage (MB/day)': data_usage
                    })

except FileNotFoundError:
    print(f"ERROR: File not found at {file_path}. Check your folder path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Requirement 3: Calculate Averages and Handle no matches
if behavior_match_count > 0:
    avg_battery = sum_battery_drain / valid_battery_count
    avg_data = sum_data_usage / valid_data_count
else:
    avg_battery = 0.0
    avg_data = 0.0
    print("Warning: No records matched the criteria.")

# Requirement 4: Export CSV
try:
    with open(exported_csv_name, mode='w', newline='', encoding='utf-8') as outfile:
        fieldnames = ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(selected_users)
    print(f"Exported {len(selected_users)} records to {exported_csv_name}")
except Exception as e:
    print(f"Error saving CSV: {e}")

# Requirement 5: Write JSON summary and Verify row count
count_verification = "Passed" if len(selected_users) == behavior_match_count else "Failed"

summary_data = {
    "operating_system_and_behavior_class": f"{target_os}, Class {target_behavior_class}",
    "total_records_in_dataset": total_records,
    "records_matching_os": os_match_count,
    "records_in_target_behavior_class": behavior_match_count,
    "average_battery_drain": round(avg_battery, 4),
    "average_data_usage": round(avg_data, 4),
    "count_verification": count_verification
}

try:
    with open(summary_json_name, mode='w', encoding='utf-8') as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)
    print(f"Summary saved to {summary_json_name}")
except Exception as e:
    print(f"Error saving JSON: {e}")

# Requirement 6: Create Backup of CSV and report size
try:
    shutil.copy(exported_csv_name, backup_csv_name)
    if os.path.exists(backup_csv_name):
        file_size = os.path.getsize(backup_csv_name)
        backup_info = f"Success. File size: {file_size} bytes"
    else:
        backup_info = "Failed. Backup file not found."
except Exception as e:
    backup_info = f"Backup failed with error: {e}"

# --- PRINT ASSESSMENT DATA ---
print("\n--- Assessment Data Results ---")
print(f"Operating system and behavior class: {target_os}, Class {target_behavior_class}")
print(f"Total records in dataset: {total_records}")
print(f"Records matching OS: {os_match_count}")
print(f"Records in target behavior class: {behavior_match_count}")
print(f"Average battery drain: {avg_battery:.4f}")
print(f"Average data usage: {avg_data:.4f}")
print(f"Count verification: {count_verification}")
print(f"Exported CSV file: {exported_csv_name}")
print(f"JSON summary file: {summary_json_name}")
print(f"Backup file and size: {backup_info}")

Target OS: iOS
Target Behavior Class: 3
Starting file processing...
Exported 251134 records to profiled_users.csv
Summary saved to usage_summary.json

--- Assessment Data Results ---
Operating system and behavior class: iOS, Class 3
Total records in dataset: 2049280
Records matching OS: 1252358
Records in target behavior class: 251134
Average battery drain: 0.9608
Average data usage: 0.1143
Count verification: Passed
Exported CSV file: profiled_users.csv
JSON summary file: usage_summary.json
Backup file and size: Success. File size: 8816879 bytes


In [9]:
#Exercise 3
import csv
import json
import shutil
import os
from collections import Counter


SEED_DIGIT = 7



target_year = 2015 + SEED_DIGIT
print(f"Target Year: {target_year}")


file_path = r"C:\Users\Gon\Lab6_File_Handling_Gonzales\household_power_consumption.txt"
filtered_csv_name = "cybersecurity_incidents_filtered.csv"
summary_json_name = "cybersecurity_summary.json"
backup_json_name = "cybersecurity_summary_backup.json"
report_txt_name = "cybersecurity_report.txt"


selected_incidents = []
total_records_in_dataset = 0
incidents_in_target_year = 0
attack_type_counter = Counter()
total_financial_loss = 0.0
valid_loss_count = 0

print("Starting file processing... (This may take a moment)")

try:
    
    with open(file_path, mode='r', encoding='utf-8', errors='ignore') as infile:
        reader = csv.DictReader(infile, delimiter=';')
        
        for row in reader:
            total_records_in_dataset += 1
            
           
            if total_records_in_dataset > 50000:
                break

            try:
                date_str = row.get('Date', '')
                if not date_str or date_str == '?':
                    continue
                    
                
                year_part = date_str.split('/')[-1]
                if int(year_part) != target_year:
                    continue
                
                
                attack_type_raw = row.get('Sub_metering_1', '0')
                attack_type = f"Type {attack_type_raw}"
                
                
                loss_str = row.get('Global_active_power', '0')
                if loss_str == '?' or not loss_str:
                    
                    continue 
                    
                financial_loss = float(loss_str)
                
                incidents_in_target_year += 1
                attack_type_counter[attack_type] += 1
                total_financial_loss += financial_loss
                valid_loss_count += 1
                
                incident_record = {
                    'Date': date_str,
                    'Time': row.get('Time', ''),
                    'Attack Type': attack_type,
                    'Financial Loss (in Million $)': financial_loss,
                    'Original_Order': incidents_in_target_year 
                }
                selected_incidents.append(incident_record)
                
            except Exception:
                
                continue

except FileNotFoundError:
    print(f"ERROR: File not found at {file_path}.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


if incidents_in_target_year > 0:
    avg_financial_loss = total_financial_loss / valid_loss_count
else:
    avg_financial_loss = 0.0
    print(f"Warning: No incidents found for the year {target_year}.")


sorted_incidents = sorted(selected_incidents, key=lambda x: x['Financial Loss (in Million $)'], reverse=True)
five_costliest = sorted_incidents[:5]


try:
    with open(filtered_csv_name, mode='w', newline='', encoding='utf-8') as outfile:
        # We exclude the 'Original_Order' from the final export
        fieldnames = ['Date', 'Time', 'Attack Type', 'Financial Loss (in Million $)']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(selected_incidents)
    print(f"Saved {len(selected_incidents)} records to {filtered_csv_name}")
except Exception as e:
    print(f"Error saving CSV: {e}")


summary_data = {
    "target_year": target_year,
    "total_records_in_dataset": total_records_in_dataset,
    "incidents_in_target_year": incidents_in_target_year,
    "count_per_attack_type": dict(attack_type_counter),
    "total_financial_loss": round(total_financial_loss, 4),
    "average_financial_loss": round(avg_financial_loss, 4),
    "five_costliest_incidents": [
        {
            "Date": inc['Date'],
            "Time": inc['Time'],
            "Attack Type": inc['Attack Type'],
            "Financial Loss": inc['Financial Loss (in Million $)']
        } for inc in five_costliest
    ]
}

try:
    with open(summary_json_name, mode='w', encoding='utf-8') as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)
    print(f"Summary saved to {summary_json_name}")
except Exception as e:
    print(f"Error saving JSON: {e}")


try:
    shutil.copy(summary_json_name, backup_json_name)
    if os.path.exists(backup_json_name):
        backup_status = f"Success. Backup created at {backup_json_name}"
    else:
        backup_status = "Failed. Backup file not found."
except Exception as e:
    backup_status = f"Backup failed with error: {e}"


report_lines = [
    "=== Cybersecurity Incident Report ===",
    f"Target Year: {target_year}",
    f"Total Records Processed: {total_records_in_dataset}",
    f"Incidents in Target Year: {incidents_in_target_year}",
    f"Total Financial Loss (USD Millions): {total_financial_loss:.4f}",
    f"Average Financial Loss (USD Millions): {avg_financial_loss:.4f}",
    "\n--- Count Per Attack Type ---"
]
for attack, count in attack_type_counter.items():
    report_lines.append(f"{attack}: {count}")

report_lines.append("\n--- Five Costliest Incidents ---")
for i, inc in enumerate(five_costliest, 1):
    report_lines.append(f"{i}. Date: {inc['Date']} | Type: {inc['Attack Type']} | Loss: ${inc['Financial Loss (in Million $)']}M")

report_lines.extend([
    "\n--- Output Files ---",
    f"Filtered CSV: {filtered_csv_name}",
    f"JSON Summary: {summary_json_name}",
    f"Backup JSON: {backup_status}"
])

try:
    with open(report_txt_name, mode='w', encoding='utf-8') as report_file:
        report_file.write("\n".join(report_lines))
    print(f"Report saved to {report_txt_name}")
except Exception as e:
    print(f"Error saving report: {e}")

# --- PRINT ASSESSMENT DATA ---
print("\n--- Assessment Data Results ---")
print(f"Target year: {target_year}")
print(f"Total records in dataset (processed): {total_records_in_dataset}")
print(f"Incidents in target year: {incidents_in_target_year}")
print(f"Count per attack type: {dict(attack_type_counter)}")
print(f"Total financial loss: {total_financial_loss:.4f}")
print(f"Average financial loss: {avg_financial_loss:.4f}")
print("Five costliest incidents:")
for inc in five_costliest:
    print(f"  - {inc['Date']} | {inc['Attack Type']} | ${inc['Financial Loss (in Million $)']}M")
print(f"Filtered CSV file: {filtered_csv_name}")
print(f"JSON summary file: {summary_json_name}")
print(f"Backup verification: {backup_status}")
print(f"Report file: {report_txt_name}")

Target Year: 2022
Starting file processing... (This may take a moment)
Saved 0 records to cybersecurity_incidents_filtered.csv
Summary saved to cybersecurity_summary.json
Report saved to cybersecurity_report.txt

--- Assessment Data Results ---
Target year: 2022
Total records in dataset (processed): 50001
Incidents in target year: 0
Count per attack type: {}
Total financial loss: 0.0000
Average financial loss: 0.0000
Five costliest incidents:
Filtered CSV file: cybersecurity_incidents_filtered.csv
JSON summary file: cybersecurity_summary.json
Backup verification: Success. Backup created at cybersecurity_summary_backup.json
Report file: cybersecurity_report.txt
